# Day 1 — 9B baseline zero-shot + 입력 축 스크리닝

**담당: 김현호·박건순. 로컬 GPU (Docker `ai2-challenge-lab` 컨테이너 안에서 연다).**

**로컬 학습은 Full 1024 하나뿐이다** — 여기서 하는 건 전부 **추론(zero-shot)** 스크리닝이다.
학습 VRAM/속도 실측은 `notebooks/vram_shutdown_probe.ipynb` /
`notebooks/full1536_crop_probe_docker.ipynb`의 몫이고, 이 노트북은 건드리지 않는다.

이 노트북은 김진영의 스키마 확정(`day1_dataset_audit.ipynb`)이 끝나야 의미 있는 결과가
나온다 — 0절에서 그걸 강제로 기다린다.

## 0. 스키마 대기 게이트 ★ 반드시 통과해야 다음 셀로 넘어간다

In [ ]:
import sys
import yaml
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

schema_report = REPO_ROOT / "data_meta" / "schema_report.md"
base_yaml = REPO_ROOT / "configs" / "base.yaml"
base_cfg = yaml.safe_load(base_yaml.read_text(encoding="utf-8"))
columns = (base_cfg.get("data", {}) or {}).get("columns", {}) or {}

schema_ready = schema_report.exists() and columns.get("image") is not None

if not schema_ready:
    print("\n" + "=" * 60)
    print("김진영의 스키마 확정을 기다리는 중입니다. MM 확인 후 다시 실행하세요.")
    print("=" * 60)
    print(f"  schema_report.md 존재: {schema_report.exists()}")
    print(f"  configs/base.yaml data.columns.image: {columns.get('image')!r}")
    raise SystemExit("스키마 대기 중 — 위 조건이 충족되면 이 셀부터 다시 실행")

print("스키마 확정 확인됨 — 계속 진행합니다.")
print("data.columns:", columns)

## 1. 환경 확인

In [ ]:
import subprocess

def nvidia_smi_query(fields):
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True, timeout=10,
    ).stdout.strip()
    return [line.split(", ") for line in out.splitlines()]

for row in nvidia_smi_query("name,memory.total,memory.free,power.limit"):
    print(f"GPU: {row[0]}  총 VRAM: {row[1]}MiB  여유: {row[2]}MiB  전력상한: {row[3]}W")

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
for pkg in ["transformers", "accelerate", "peft", "bitsandbytes", "fla", "triton"]:
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"{pkg}: 설치 안 됨")

from src import resources
kernel_impl = resources.kernel_impl_snapshot()
print("\nkernel_impl:", kernel_impl)
if kernel_impl.get("gated_delta_rule_impl") != "kernel":
    print("*** 경고: gated_delta_rule_impl이 'kernel'이 아닙니다 — torch_fallback이면 sec/step이 훨씬 느립니다 ***")

In [ ]:
# --- 9B 가중치가 로컬 캐시에 있는지 확인 ---
import os

hf_home = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
model_cache = Path(hf_home) / "hub" / "models--Qwen--Qwen3.5-9B"

if not model_cache.exists():
    print(f"\n*** 9B 가중치가 로컬 캐시({model_cache})에 없습니다. ***")
    print("*** 먼저 `python tools/prefetch_weights.py`를 실행해서 받아두세요 (주말 작업 대상). ***")
    raise SystemExit("9B 가중치 없음 — prefetch 먼저 실행")

print(f"9B 가중치 캐시 확인됨: {model_cache}")

## 2. baseline zero-shot ★ 팀 전체의 기준점

`infer` 커맨드는 샘플 수를 `--set subset.size=300`으로 지정한다(`smoke`는 로드/토큰/PEFT
왕복 점검용으로 샘플 1개만 보므로 Accuracy 기준점에 쓸 수 없다 — 반드시 `infer`를 쓴다).
타입 컬럼이 있으면 `subset.stratify_by=type`로 유형별 비율을 유지한 채 뽑는다.

**실패(ERROR/OOM)해도 예외를 던지지 않는다** — `run_infer`는 항상 `status`/`notes`가
채워진 dict를 돌려주고, 아래 셀들이 그걸 그대로 `day1_input_axis.csv`에 append한다.
박건순 PC가 `exit=1`로 죽었을 때 셀 출력에만 에러가 찍히고 CSV에는 안 남아서 원인을
못 잡았던 문제를 막기 위한 것 — **CSV의 `notes` 컬럼에 마지막 400자가 항상 남는다.**

In [ ]:
import json, time

def existing_run_dirs():
    runs_dir = REPO_ROOT / "runs"
    return set(runs_dir.iterdir()) if runs_dir.exists() else set()

def _classify_status(returncode, output):
    if returncode == 0:
        return "ok"
    if "OutOfMemoryError" in output or "CUDA out of memory" in output:
        return "OOM"
    return f"ERROR(exit={returncode})"

def run_infer(config_name, extra_sets, split="dev", n=300, timeout_sec=1800):
    """항상 dict를 돌려준다 — 성공/실패 모두 status/notes/run_dir/elapsed_sec를 포함한다."""
    before = existing_run_dirs()
    args = ["infer", "-c", config_name, "--split", split,
            "--set", f"subset.size={n}", "--set", "subset.stratify_by=type", *extra_sets]
    t0 = time.time()
    try:
        proc = subprocess.run([sys.executable, "-m", "src.run", *args],
                              cwd=str(REPO_ROOT), capture_output=True, text=True, timeout=timeout_sec)
        elapsed = time.time() - t0
        output = (proc.stdout or "") + (proc.stderr or "")
        status = _classify_status(proc.returncode, output)
    except subprocess.TimeoutExpired as exc:
        elapsed = time.time() - t0
        output = ((exc.stdout or "") + (exc.stderr or "")) if hasattr(exc, "stdout") else ""
        status = "TIMEOUT"

    after = existing_run_dirs()
    new_dirs = after - before
    run_dir = sorted(new_dirs)[-1] if new_dirs else None

    result = {"status": status, "notes": "", "run_dir": run_dir, "elapsed_sec": round(elapsed, 1)}
    if status != "ok":
        result["notes"] = (output[-400:] if output else "(서브프로세스 출력 없음)")
        print(f"  -> {status}")
        print(output[-2000:] if output else "(출력 없음)")
        return result

    summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
    result.update(summary["infer"])
    return result

In [ ]:
baseline_result = run_infer("9b_base_1024", [])
print("=== baseline: 9b_base_1024, dev split, n=300 ===")
print(json.dumps(baseline_result, ensure_ascii=False, indent=2, default=str))

if baseline_result["status"] != "ok":
    print("\n*** baseline이 실패했습니다 — 아래 3절 축 스크리닝도 같은 실패가 반복될 가능성이 높습니다. ***")
else:
    vt = baseline_result.get("visual_tokens") or {}
    if vt.get("median") not in (None, 768):
        print(f"\n*** 경고: visual_tokens median={vt.get('median')} — 768이 아닙니다! ***")
        print("*** 「06」 24-1절의 단위 함정을 확인하세요(원본 픽셀이 아니라 processor 실측 토큰 기준). ***")

## 3. 입력 축 zero-shot 스크리닝

**학습이 아니라 추론이다.** 한 번에 한 축만 바꾼다: long_side=1536, long_side=2048,
crops=4(crop_long_side=768). 각 설정마다 Accuracy·VRAM·sec/sample을 한 행씩
`reports/comparisons/day1_input_axis.csv`에 append한다.

In [ ]:
AXIS_CONFIGS = [
    ("baseline(long_side=1024)", []),
    ("long_side=1536", ["--set", "input.long_side=1536"]),
    ("long_side=2048", ["--set", "input.long_side=2048"]),
    ("crops=4,crop_long_side=768", ["--set", "input.crops=4", "--set", "input.crop_long_side=768"]),
]

import csv

OUT_CSV = REPO_ROOT / "reports" / "comparisons" / "day1_input_axis.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
FIELDS = ["Eval N", "config", "axis", "Best Input", "run_id", "accuracy", "peak_vram_gb",
          "sec_per_sample", "visual_tokens_median", "status", "notes"]

def append_axis_row(row):
    is_new = not OUT_CSV.exists()
    with open(OUT_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if is_new:
            w.writeheader()
        w.writerow({k: row.get(k, "") for k in FIELDS})
        f.flush(); os.fsync(f.fileno())

# 첫 항목(baseline)은 위에서 이미 돌렸으므로 재사용하고, 나머지 축만 새로 돌린다.
axis_rows = []
for axis_name, extra_sets in AXIS_CONFIGS:
    if axis_name.startswith("baseline"):
        result = baseline_result
    else:
        print(f"\n=== axis: {axis_name} ===")
        result = run_infer("9b_base_1024", extra_sets)

    vt = result.get("visual_tokens") or {}
    run_dir = result.get("run_dir")
    row = {
        "Eval N": result.get("n"),
        "config": "9b_base_1024",
        "axis": axis_name,
        "Best Input": f"max_visual_tokens={vt.get('max')}" if vt else "",
        "run_id": run_dir.name if run_dir else "",
        "accuracy": result.get("accuracy"),
        "peak_vram_gb": result.get("peak_vram_gb"),
        "sec_per_sample": result.get("sec_per_sample"),
        "visual_tokens_median": vt.get("median"),
        "status": result.get("status"),
        "notes": result.get("notes", ""),
    }
    axis_rows.append(row)
    append_axis_row(row)
    print(f"  [{axis_name}] status={row['status']}  accuracy={row['accuracy']}  "
          f"peak_vram_gb={row['peak_vram_gb']}  sec/sample={row['sec_per_sample']}")

print(f"\n저장됨: {OUT_CSV}")

## 4. 「11」에 붙일 표 출력

In [ ]:
print("| 설정 | N | Accuracy | Peak VRAM (GB) | sec/sample | Best Input | status |")
print("|---|---|---|---|---|---|---|")
for row in axis_rows:
    print(f"| {row['axis']} | {row['Eval N']} | {row['accuracy']} | {row['peak_vram_gb']} | "
          f"{row['sec_per_sample']} | {row['Best Input']} | {row['status']} |")

failed = [r for r in axis_rows if r["status"] != "ok"]
if failed:
    print("\n실패한 축이 있습니다 — reports/comparisons/day1_input_axis.csv의 notes 컬럼에 마지막 400자가 남아 있습니다:")
    for row in failed:
        print(f"  [{row['axis']}] {row['status']}: {row['notes'][:200]}")